In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

In [3]:
import requests



In [4]:
# Regafi API helper functions

def regafi_request(path, method='GET', params=None, data=None, headers=None, base_url='https://developer.regafi.banque-france.fr'):
    """Simple helper to call the Regafi API. Returns parsed JSON or None on error.
    - path: API path (e.g. '/v1/entities')
    - method: 'GET' or 'POST'
    - params: dict for query string
    - data: dict for POST body
    - headers: dict for additional headers (e.g. {'Authorization': 'Bearer ...'})
    - base_url: override if required
    """
    url = base_url.rstrip('/') + '/' + path.lstrip('/')
    hdrs = {'Accept': 'application/json'}
    if headers:
        hdrs.update(headers)
    try:
        if method.upper() == 'GET':
            r = requests.get(url, params=params, headers=hdrs, timeout=30)
        else:
            r = requests.post(url, json=data, params=params, headers=hdrs, timeout=30)
        r.raise_for_status()
        return r.json()
    except requests.RequestException as e:
        print('Regafi request error:', e)
        return None

# Example usage:
# api_key = 'YOUR_API_KEY'  # if the API requires a key
# headers = {'Authorization': f'Bearer {api_key}'} if api_key else None
# resp = regafi_request('/v1/entities', params={'q': 'bank'}, headers=headers)
# if resp:
#     # adapt to the response schema, common pattern: results list or top-level list
#     data_list = resp.get('results', resp)
#     df_api = pd.json_normalize(data_list)
#     print(df_api.head())


In [5]:
regafi_request('https://developer.regafi.banque-france.fr/')

Regafi request error: Expecting value: line 1 column 1 (char 0)


In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'FR ACP' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running FR ACP Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName + ' 1': 'https://www.regafi.fr/spip.php?rubrique3',

        }



Typology={

        regulatorName+' 1': 'Registered entities',


        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': []}



searchValues = list(string.ascii_lowercase) + list(map(str, range(10)))

now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    
    
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(2)
    advanced_search = soup.find('div',class_='contentHome')
    advanced_search = advanced_search.find('h2',class_='advancedsearch').parent['href']
    driver.get('https://www.regafi.fr/'+advanced_search)
    sleep(2)
    # soup2 = BeautifulSoup(driver.page_source, 'html.parser') 
    # submit_button = driver.find_element(By.XPATH, '//input[@type="submit"]')
    # driver.execute_script("arguments[0].click();", submit_button)
    # sleep(2)
    whole_list_url = 'https://www.regafi.fr/spip.php?page=results&type=advanced&id_secteur=3&lang=en&denomination=&siren=&cib=&bic=&nom=&siren_agent=&num=&cat=0&retrait=0'
    sleep(2)
    driver.get(whole_list_url)
    soup2 = BeautifulSoup(driver.page_source, 'html.parser') 
    sleep(2)
    # download the csv file
    export_url = soup2.find(
        'span',class_='exportresults'
    ).find('a')['href']
    driver.get('https://www.regafi.fr/'+export_url)
    sleep(2)
    csv_file = os.listdir(tempfolder)[0]
    if csv_file:
        csv_path = os.path.join(tempfolder, csv_file)
        try:
            df = pd.read_csv(csv_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(csv_path, encoding='latin1')
        sleep(1)
    driver.quit()
    df = df.fillna('')
    sqldict = bourange_same_length_array(sqldict)   
    df_sql=pd.DataFrame(sqldict)
    df_sql['InternalID_1'] = df['REGAFI identifier']
    df_sql['InternalID_1_type'] = 'REGAFI identifier'
    df_sql['InternalID_2'] = df['National identifier (SIREN number for French entities)']
    df_sql['InternalID_2_type'] = 'National identifier (SIREN number for French entities)'
    df_sql['Name'] = df['Name']
    df_sql['LEI Code'] = df['LEI']
    df_sql['BIC SWIFT Code'] = df['Bank code (CIB)']
    df_sql['ListProcessDate'] = processdate
    df_sql['RegCtry'] = reg.split(' ')[0]
    df_sql['RegCode'] = reg.split(' ')[1]
    df_sql['ListCode'] = reg.split(' ')[-1]
    df_sql['ListName'] = Typology[reg]
    df_sql['RegulationType'] = 'Regulated'   

    df_sql = df_sql.fillna('')

        
    


    

    


Working with list FR ACP 1


AttributeError: 'NoneType' object has no attribute 'find'

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

# df=pd.DataFrame(sqldict)

# df = df.drop_duplicates()

df_sql.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_24092\690050426.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [21]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert
regulatorName = "FR ACP"

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import camelot
url = "https://acpr.banque-france.fr/fr/professionnels/lacpr-vous-accompagne/banque/decouvrir-le-controle-bancaire/entites-systemiques-du-secteur-bancaire#Rle-de-lACPR-et-coordination-avec-les-autres-autorits-79227"

html = requests.get(url, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

h2 = soup.find("h2", id="Entits-systmiques-du-secteur-bancaire-79229")
if not h2:
    raise SystemExit("h2 not found")

div = h2.find_next_sibling("div")
if not div:
    raise SystemExit("sibling div not found")

table = div.find("table")
if not table:
    raise SystemExit("table not found")

target_row = None
for tr in table.find_all("tr"):
    if "listes officielles d'entités et coussins associés en france" in tr.get_text(" ", strip=True).lower():
        target_row = tr
        break

if not target_row:
    raise SystemExit("target row not found")

tds = target_row.find_all("td")
if len(tds) < 2:
    raise SystemExit("not enough tds in target row")

first_link = tds[0].find_all("a")[-1] if tds[0].find_all("a") else None
second_link = tds[1].find_all("a")[-1] if tds[1].find_all("a") else None

if not first_link or not second_link:
    raise SystemExit("could not find last links in both tds")

links = [urljoin(url, first_link.get("href")), urljoin(url, second_link.get("href"))]

pdf_links = []
for link in links:
    page_html = requests.get(link, timeout=30).text
    page_soup = BeautifulSoup(page_html, "html.parser")
    a = page_soup.find("a", attrs={"role": "button", "data-file-extension": "pdf"})
    h1 = page_soup.find("h1").text.strip() if page_soup.find("h1") else ""
    if not a or not a.get("href"):
        raise SystemExit(f"pdf button link not found in {link}")
    pdf_links.append(urljoin(link, a.get("href")))

print("PDF links:")
for p in pdf_links:
    print(p)
    sleep(2)
    driver.get(p)
    
    print("PDF Title:", h1)
        

    print(os.listdir(tempfolder))
    sleep(2)
    pdf_file = os.listdir(tempfolder)[0]
    filePath = os.path.join(tempfolder, pdf_file)

    tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       

    for k in range(tables.n):
        df_Table = tables[k].df
        
        for j in range(1,len(df_Table)):
                name = ' '.join([item.strip() for item in df_Table[0][j].splitlines() if item !=''])
                name = name.replace('*', ' ')
                address = ' '.join([item.strip() for item in df_Table[1][j].splitlines() if item !=''])
                print(f"Name: {name}, Address: {address}, Zip Code: {find_zip_code(address)}")
    os.remove(filePath)


PDF links:
https://acpr.banque-france.fr/system/files/2025-12/20251201_Liste_EISm_2025_au_titre_2024.pdf
PDF Title: Liste des Autres établissements d’importance systémique (A-EIS) au titre de l’exercice 2024 conformément aux dispositions de l'article L511-41-1 A VII du Code monétaire et financier
['59e42e75-81c4-4ac6-aabe-472afdd8a558.tmp']
Name: , Address: , Zip Code: 
Name: BNP PARIBAS, Address: 16 boulevard des Italiens 75009 Paris, Zip Code: 75009
Name: GROUPE CRÉDIT AGRICOLE, Address: 12 Place des États-Unis 92120 Montrouge, Zip Code: 92120
Name: SOCIÉTÉ GÉNÉRALE, Address: 17 cours Valmy 92972 Paris La Défense, Zip Code: 92972
Name: GROUPE BPCE, Address: 7 Promenade Germaine- Sablon 75013 Paris, Zip Code: 75013
https://acpr.banque-france.fr/system/files/2025-12/20251201_Liste_AEIS_2025_au_titre_2024.pdf
PDF Title: Liste des Autres établissements d’importance systémique (A-EIS) au titre de l’exercice 2024 conformément aux dispositions de l'article L511-41-1 A VII du Code monétaire 

In [16]:

def find_zip_code(string):
    match = re.search(r'\d{5}', string)
    if match:
        return match.group()
    else:
        return ''

In [17]:

import camelot
print(os.listdir(tempfolder))
pdf_file = os.listdir(tempfolder)[0]
filePath = os.path.join(tempfolder, pdf_file)

tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       

for k in range(tables.n):
    df_Table = tables[k].df
    
    for j in range(1,len(df_Table)):
            name = ' '.join([item.strip() for item in df_Table[0][j].splitlines() if item !=''])
            name = name.replace('*', ' ')
            address = ' '.join([item.strip() for item in df_Table[1][j].splitlines() if item !=''])
            print(f"Name: {name}, Address: {address}, Zip Code: {find_zip_code(address)}")

['20251201_Liste_AEIS_2025_au_titre_2024.pdf', '20251201_Liste_EISm_2025_au_titre_2024.pdf']
Name: BNP PARIBAS  , Address: 16 boulevard des Italiens 75009 Paris, Zip Code: 75009
Name: GROUPE CRÉDIT AGRICOLE  , Address: 12 Place des États-Unis 92120 Montrouge, Zip Code: 92120
Name: SOCIÉTÉ GÉNÉRALE  , Address: 17 cours Valmy 92972 Paris La Défense, Zip Code: 92972
Name: GROUPE BPCE  , Address: 7 Promenade Germaine- Sablon 75013 Paris, Zip Code: 75013
Name: GROUPE CRÉDIT MUTUEL, Address: 46 rue du Bastion 75017 Paris, Zip Code: 75017
Name: HSBC CE, Address: 38 avenue Kléber 75116 Paris, Zip Code: 75116
Name: LA BANQUE POSTALE, Address: 115 rue de Sèvres 75006 Paris, Zip Code: 75006


In [26]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://acpr.banque-france.fr/fr/professionnels/vos-outils-et-services/consulter-les-registres/registre-des-agents-financiers-et-des-organismes-dassurance"

html = requests.get(url, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

h3 = None
for tag in soup.find_all("h3"):
    if "télécharger la liste des organismes d'assurance actifs".lower() in tag.get_text(" ", strip=True).lower():
        h3 = tag
        break

if not h3:
    raise SystemExit("h3 not found")

div = h3.find_next_sibling("div")
if not div:
    raise SystemExit("sibling div not found")

links = div.find_all("a", attrs={"role": "button", "data-file-extension": "xlsx"})
if not links:
    raise SystemExit("no matching xlsx links found")

last_link = links[-1].get("href")
if not last_link:
    raise SystemExit("last link missing href")

final_url = urljoin(url, last_link)
print(final_url)
driver.get(final_url)

https://acpr.banque-france.fr/system/files/2026-03/20260302_liste_des_organismes_d_assurances_actifs.xlsx


In [ ]:
print(os.listdir(tempfolder))
sleep(2)
pdf_file = os.listdir(tempfolder)[0]
filePath = os.path.join(tempfolder, pdf_file)
sleep(1)
df_list_3 = pd.read_excel(filePath)
for index, row in df_list_3.iterrows():
    id_ = row.iloc[0]
    name_ = row.iloc[1]
    topology = row.iloc[2]
    address_ = row.iloc[8]
    code_zip = row.iloc[9]
    city_ = row.iloc[10] 
    lei = row.iloc[12]
    

        

['20260302_liste_des_organismes_d_assurances_actifs.xlsx']


200081
220015
220017
220070
220080
220090
220092
220125
220148
220178
220183
220207
220218
220232
220262
220268
220288
220296
220336
220357
220361
220363
220379
220381
220418
220430
220437
220450
220474
220486
220503
220507
220538
220547
220565
220576
220582
220597
220617
220634
220638
220650
220666
220676
220839
220842
220845
220846
220870
220874
220883
220905
220920
220934
220982
221026
221034
221035
221051
221054
221068
221087
221119
221124
221138
221155
221201
221226
221264
221266
221289
221343
221361
221372
221380
221394
221395
221426
221430
221441
221465
221467
221525
221551
221562
221586
221620
221627
221634
221665
221685
221716
221728
221879
221913
221916
221936
221952
221975
221980
221992
222043
222096
222143
222219
222227
222273
222274
222283
222384
222412
222419
222428
222499
222527
222531
222557
222572
222576
222611
222613
222618
222638
222658
222667
222673
222690
222727
222740
222759
222792
222798
222799
222810
222814
222904
222906
222909
222929
222945
222962
222967
222994

In [37]:
row.iloc[0]

31080004